In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import spacy
import os
import importlib

import utils
from gpt_constraining import generate_with_selection_scored
from utils import DATA_DIR, select_gpu
# importlib.reload(gpt_constraining)

gpu_id = select_gpu()
spacy.require_gpu(gpu_id)
nlp = spacy.load("nl_core_news_md", disable=["ner", "textcat", "lemmatizer"])
print("Loaded spacy model.")


Selected GPU 0 with 454.62 MB used memory.
Loaded spacy model.


In [3]:
model = AutoModelForCausalLM.from_pretrained("GroNLP/gpt2-small-dutch")
print("Loaded GPT2 model.")
tokenizer = AutoTokenizer.from_pretrained("GroNLP/gpt2-small-dutch")
print("Loaded GPT2 tokenizer.")

Loaded GPT2 model.
Loaded GPT2 tokenizer.


In [5]:
# we load in the tensors from previous experiments
import pickle
with open(DATA_DIR/"tensors/fineweb_dutch_vectors_ids/decomposition/counting_1000d_100r_1000i.pkl", "rb") as f:
    core, factors = pickle.load(f)

# we import the vocab mapping
with open(DATA_DIR/"tensors/fineweb_dutch_vectors_ids/vocabularies_1000.pkl", "rb") as f:
    vocab = torch.load(f)
# we need to move them to the correct device
fcore = core.to(f"cuda:{gpu_id}")
factors = [factor.to(f"cuda:{gpu_id}") for factor in factors]
vocab_v = vocab["vocab_v"]
vocab_s = vocab["vocab_s"]
vocab_o = vocab["vocab_o"]
v2i = vocab["v2i"]
s2i = vocab["s2i"]
o2i = vocab["o2i"]

In [6]:
# we print dimension 100 from the verb factors
verb_factors = factors[0]
i = 46

verb_factors_i = verb_factors[:, i]
# we normalize all values according to the dimension
scores, indices = torch.topk(verb_factors[:, i], len(verb_factors_i))
# we construct scored dict
score_dict = {vocab_v[index]:score for index, score in zip(indices, scores)}
# we print the top k best entries
print(list(score_dict.items())[:10])
# alternatively, give rank-based scores
# we make a reranked score list
scores_reranked = []
for i, score in enumerate(scores):
    if i < 5:
        scores_reranked.append(1.0)
    elif i < 15:
        scores_reranked.append(0.5)
    elif i < 30:
        scores_reranked.append(0.25)
    else:
        scores_reranked.append(0.0)
#scores_reranked = [max((100-2*i)/100, 0) for (i, score) in enumerate(scores)]
score_dict_reranked = {vocab_v[index]:score for index, score in zip(indices, scores_reranked) if not score==0}
print(list(score_dict_reranked.items())[:10])

[('blijken', tensor(3.4053e-07, device='cuda:0')), ('gaan', tensor(2.6628e-07, device='cuda:0')), ('lijken', tensor(2.0542e-07, device='cuda:0')), ('noemen', tensor(1.8954e-07, device='cuda:0')), ('geboren', tensor(1.7842e-07, device='cuda:0')), ('staan', tensor(1.7767e-07, device='cuda:0')), ('kijken', tensor(1.7429e-07, device='cuda:0')), ('uitvoeren', tensor(1.5895e-07, device='cuda:0')), ('brengen', tensor(1.5356e-07, device='cuda:0')), ('aanbevelen', tensor(1.3132e-07, device='cuda:0'))]
[('blijken', 1.0), ('gaan', 1.0), ('lijken', 1.0), ('noemen', 1.0), ('geboren', 1.0), ('staan', 0.5), ('kijken', 0.5), ('uitvoeren', 0.5), ('brengen', 0.5), ('aanbevelen', 0.5)]


In [7]:
# we now generate a sentence with only words from this dimension
print(generate_with_selection_scored(model, tokenizer, "Hij pakt zijn telefoon op en",
                               score_dict, max_length=100, repeat=True))

Picked word:  gaat
Picked word:  's
Picked word:  geweest
Picked word:  omkomend
Picked word:  optraden
Picked word:  metende
Picked word: Is
Picked word:  vraagt
Picked word:  is
Hij pakt zijn telefoon op en gaat verder. 's Middags wordt hij thuis bij de huisarts aan het ziekenhuis in Rotterdam waar ik ook fysiotherapeut ben geweest omkomend optraden, metende ogen van pijn:
'Is er iets wat je te goed doet als een patiënt overlijdt?' vraagt dokter Marvin Dernes na afloop haar hand bezorgd af terwijl ze naar buiten wijst over hoe zij zelf reageert toen we hun laatste reanimatie hadden ondergaan! Ze is blij dat Rob Maas heeft geholpen; misschien wil iemand anders nog eens zien of wij kunnen helpen? Ik


In [12]:
print(generate_with_selection_scored(model, tokenizer, "Hij pakt zijn telefoon op en", score_dict_reranked,
                                    temperature=0.7, max_length=100, repeat=True))

No info for speel found in lexicon
No info for scheelen found in lexicon
No info for schenk found in lexicon
No info for drink found in lexicon
No info for zing found in lexicon
No info for meegemaken found in lexicon
Picked word:  kijkt
No info for speel found in lexicon
No info for scheelen found in lexicon
No info for schenk found in lexicon
No info for drink found in lexicon
No info for zing found in lexicon
No info for meegemaken found in lexicon
Picked word:  Gegeten
No info for speel found in lexicon
No info for scheelen found in lexicon
No info for schenk found in lexicon
No info for drink found in lexicon
No info for zing found in lexicon
No info for meegemaken found in lexicon
Picked word:  kijkt
No info for speel found in lexicon
No info for scheelen found in lexicon
No info for schenk found in lexicon
No info for drink found in lexicon
No info for zing found in lexicon
No info for meegemaken found in lexicon
Picked word:  kijkt
No info for speel found in lexicon
No info for